In [ ]:
load_ext jupyter_black

In [ ]:
import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS
import statsmodels.formula.api as smf
from sklearn.preprocessing import minmax_scale

In [ ]:
questions = pd.read_pickle("data/Llama-3.1-8B-Instruct_questions.gz")
questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
questions_baseline_answers = questions[["q_id", "baseline_answer"]]

questions_baseline_answers.loc[
    questions_baseline_answers["q_id"].isin([f"q_{i}" for i in range(121, 151)]),
    "baseline_answer",
] = (
    questions_baseline_answers.loc[
        questions_baseline_answers["q_id"].isin([f"q_{i}" for i in range(121, 151)]),
        "baseline_answer",
    ]
    .str.replace(",", "")
    .str.extract(r"^[^\d]*(\d+)", expand=False)
    .astype(float)
)

questions_baseline_answers = pd.Series(
    questions_baseline_answers.baseline_answer.values,
    index=questions_baseline_answers.q_id,
).to_dict()

In [ ]:
concepts = {
    "prism": [
        # (
        #    "salary",
        #    "topic:travel,itinerary,trip",
        # ),
        # ("salary", "e_gratitude_user_prompt"),
        ("medical", "s_negative_user_prompt"),
    ]
    # "prism": ["age", "pride_model", "recipe"],
    # "cad_en": ["gender", "fantasy", "disappointment_user"],
}

In [ ]:
for dataset in concepts:
    for domain, concept in concepts[dataset]:
        id_col = {
            "prism": "conversation_id",
            "cad_en": "conversation_id",
        }

        demographics = {
            "prism": [
                "age",
                "gender",
                "employment_status",
                "education",
                "marital_status",
                "english_proficiency",
                "religion",
                "ethnicity",
                "birth_region",
                "reside_region",
                "lm_familiarity",
            ],
            "cad_en": [
                "annotator_age",
                "annotator_gender",
                "annotator_education_level",
                "annotator_political",
                "annotator_ethnicity",
            ],
        }
        df = pd.read_pickle(
            f"llama_erasure/Llama-3.1-8B-Instruct_{dataset}_dim_{domain}_{concept}_answers.gz"
        )
        qrange = {
            "benefits": (61, 91),
            "political": (91, 121),
            "salary": (121, 151),
            "legal": (151, 181),
            "medical": (181, 211),
        }[domain]
        if domain != "salary":
            for c in [f"q_{i}" for i in range(*qrange)]:
                df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
        else:
            for c in [f"q_{i}" for i in range(121, 151)]:
                df[c] = (
                    df[c]
                    .str.replace(",", "")
                    .str.extract(r"^[^\d]*(\d+)")
                    .astype(float)
                )

        # df["accuracy"] = df[[f"q_{i}" for i in range(50)]].mean(axis=1)
        df[domain] = df[[f"q_{i}" for i in range(*qrange)]].mean(axis=1)

        df = df.drop(columns=[f"q_{i}" for i in range(*qrange)])

        df_linguistic = pd.read_pickle(
            f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
        ).drop(
            columns=["s_neutral_model_response", "s_neutral_user_prompt"],
            errors="ignore",
        )
        for c in ["politeness_user_prompt", "politeness_model_response"]:
            if c in df_linguistic:
                df_linguistic[c] = df_linguistic[c].replace(
                    {
                        "impolite": 0,
                        "neutral": 0.5,
                        "polite": 1,
                        "somewhat polite": 0.75,
                    }
                )
        df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
        if dataset == "prism":
            demographics[dataset] += ["model_name"]
        demographics[dataset] += ["topic"]
        group_cols = [id_col[dataset]] + demographics[dataset]
        df_linguistic = (
            df_linguistic.groupby(group_cols)[
                [
                    c
                    for c in df_linguistic.columns
                    if "_model_response" in c or "_user_prompt" in c
                ]
            ]
            .mean()
            .reset_index()
        )
        df = df.merge(
            df_linguistic[
                [id_col[dataset]]
                + [
                    c
                    for c in df_linguistic.columns
                    if "_model_response" in c
                    or "_user_prompt" in c
                    or c == "topic"
                    or c == "model_name"
                ]
            ],
            on=id_col[dataset],
        )
        demographics[dataset] += [
            c for c in df.columns if "_model_response" in c or "_user_prompt" in c
        ]

        df_beliefs = pd.read_pickle(
            f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
        )
        df_beliefs.columns = df_beliefs.columns.str.replace(" ", "")
        cols = [
            c
            for c in df_beliefs.columns
            if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
        ] + ["revealed_Gender"]
        if "human_Gender" in df_beliefs.columns:
            cols += ["human_Gender"]
        demographics[dataset] += cols
        cols.append(id_col[dataset])
        df = df.merge(df_beliefs[cols], on=id_col[dataset])

        # df[
        #     [
        #         c
        #         for c in demographics[dataset]
        #         if "_model_response" in c or "_user_prompt" in c
        #     ]
        # ] = minmax_scale(
        #     df[
        #         [
        #             c
        #             for c in demographics[dataset]
        #             if "_model_response" in c or "_user_prompt" in c
        #         ]
        #     ]
        # )

        cols = [domain]

        for col in cols:
            filtered_df = df.loc[~df[col].isna()]
            if not os.path.isfile(
                f"figures_leace_regression/{dataset}_{concept}_{col}_correct_1.png"
            ):
                demo_cols = [
                    c
                    for c in demographics[dataset]
                    if "_model_response" in c
                    or "_user_prompt" in c
                    or c == "model_name"
                ]
                mod = smf.ols(
                    formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                )
                res = mod.fit()
                result_df = pd.read_html(
                    res.summary().tables[1].as_html(), header=0, index_col=0
                )[0].reset_index()
                fig = plt.figure(figsize=(6.5, 5))
                ax = sns.barplot(
                    result_df.loc[
                        (result_df["P>|t|"] < 0.05)
                        & (result_df["index"] != "Intercept")
                    ].sort_values(by="coef"),
                    x="index",
                    y="coef",
                )
                ax.tick_params(axis="x", labelrotation=90)
                fig.savefig(
                    f"figures_leace_regression/{dataset}_{concept}_{col}_correct_1.png",
                    bbox_inches="tight",
                )
                plt.show()
                print(res.summary())
            if "topic" in filtered_df:
                filtered_df = filtered_df.loc[
                    filtered_df.duplicated(subset=["topic"], keep=False)
                ]
            if not os.path.isfile(
                f"figures_leace_regression/{dataset}_{concept}_{col}_correct_2.png"
            ):
                demo_cols = [
                    c
                    for c in demographics[dataset]
                    if "_model_response" in c
                    or "_user_prompt" in c
                    or c == "topic"
                    or c == "model_name"
                ]
                mod = smf.ols(
                    formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                )
                res = mod.fit()
                result_df = pd.read_html(
                    res.summary().tables[1].as_html(), header=0, index_col=0
                )[0].reset_index()
                result_df["linguistic"] = ~result_df["index"].str.contains("topic")
                fig = plt.figure(figsize=(40, 5))
                ax = sns.barplot(
                    result_df.loc[
                        (result_df["P>|t|"] < 0.05)
                        & (result_df["index"] != "Intercept")
                    ].sort_values(by="coef"),
                    x="index",
                    y="coef",
                    hue="linguistic",
                )
                ax.tick_params(axis="x", labelrotation=90)
                fig.savefig(
                    f"figures_leace_regression/{dataset}_{concept}_{col}_correct_2.png",
                    bbox_inches="tight",
                )
                plt.show()
                print(res.summary())

            for demographic in [
                "age",
                "gender",
                "education",
                "ethnicity",
                "religion",
                "english",
                "marital",
            ]:
                if not os.path.isfile(
                    f"figures_leace_regression/{dataset}_{concept}_{col}_{demographic}_correct_3.png"
                ):
                    demo_cols = [
                        c
                        for c in demographics[dataset]
                        if demographic in c.lower()
                        or "_model_response" in c
                        or "_user_prompt" in c
                        or c == "topic"
                        or c == "model_name"
                    ]

                    mod = smf.ols(
                        formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                    )
                    res = mod.fit()
                    result_df = pd.read_html(
                        res.summary().tables[1].as_html(), header=0, index_col=0
                    )[0].reset_index()
                    result_df["type"] = result_df["index"].str.extract("(topic)")
                    result_df["type"].loc[
                        result_df["index"].str.contains(demographic)
                    ] = "demographic"
                    result_df["type"].loc[result_df["type"].isna()] = "linguistic"
                    fig = plt.figure(figsize=(45, 5))
                    ax = sns.barplot(
                        result_df.loc[
                            (result_df["P>|t|"] < 0.05)
                            & (result_df["index"] != "Intercept")
                        ].sort_values(by="coef"),
                        x="index",
                        y="coef",
                        hue="type",
                    )
                    ax.tick_params(axis="x", labelrotation=90)
                    fig.savefig(
                        f"figures_leace_regression/{dataset}_{concept}_{col}_{demographic}_correct_3.png",
                        bbox_inches="tight",
                    )
                    plt.show()
                    print(res.summary())

In [ ]:
for dataset in concepts:
    for domain, concept in concepts[dataset]:
        id_col = {
            "prism": "conversation_id",
            "cad_en": "conversation_id",
        }

        demographics = {
            "prism": [
                "age",
                "gender",
                "employment_status",
                "education",
                "marital_status",
                "english_proficiency",
                "religion",
                "ethnicity",
                "birth_region",
                "reside_region",
                "lm_familiarity",
            ],
            "cad_en": [
                "annotator_age",
                "annotator_gender",
                "annotator_education_level",
                "annotator_political",
                "annotator_ethnicity",
            ],
        }
        df = pd.read_pickle(
            f"llama_erasure/Llama-3.1-8B-Instruct_{dataset}_dim_{domain}_{concept}_answers.gz"
        )
        qrange = {
            "benefits": (61, 91),
            "political": (91, 121),
            "salary": (121, 151),
            "legal": (151, 181),
            "medical": (181, 211),
        }[domain]
        if domain != "salary":
            for c in [f"q_{i}" for i in range(*qrange)]:
                df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
        else:
            for c in [f"q_{i}" for i in range(121, 151)]:
                df[c] = (
                    df[c]
                    .str.replace(",", "")
                    .str.extract(r"^[^\d]*(\d+)")
                    .astype(float)
                )

        # df["accuracy"] = df[[f"q_{i}" for i in range(50)]].mean(axis=1)
        df[domain] = df[[f"q_{i}" for i in range(*qrange)]].mean(axis=1)

        df = df.drop(columns=[f"q_{i}" for i in range(*qrange)])

        df_linguistic = pd.read_pickle(
            f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
        ).drop(
            columns=["s_neutral_model_response", "s_neutral_user_prompt"],
            errors="ignore",
        )
        for c in ["politeness_user_prompt", "politeness_model_response"]:
            if c in df_linguistic:
                df_linguistic[c] = df_linguistic[c].replace(
                    {
                        "impolite": 0,
                        "neutral": 0.5,
                        "polite": 1,
                        "somewhat polite": 0.75,
                    }
                )
        df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
        if dataset == "prism":
            demographics[dataset] += ["model_name"]
        demographics[dataset] += ["topic"]
        group_cols = [id_col[dataset]] + demographics[dataset]
        df_linguistic = (
            df_linguistic.groupby(group_cols)[
                [
                    c
                    for c in df_linguistic.columns
                    if "_model_response" in c or "_user_prompt" in c
                ]
            ]
            .mean()
            .reset_index()
        )
        df = df.merge(
            df_linguistic[
                [id_col[dataset]]
                + [
                    c
                    for c in df_linguistic.columns
                    if "_model_response" in c
                    or "_user_prompt" in c
                    or c == "topic"
                    or c == "model_name"
                ]
            ],
            on=id_col[dataset],
        )
        demographics[dataset] += [
            c for c in df.columns if "_model_response" in c or "_user_prompt" in c
        ]

        df_beliefs = pd.read_pickle(
            f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
        )
        df_beliefs.columns = df_beliefs.columns.str.replace(" ", "")
        cols = [
            c
            for c in df_beliefs.columns
            if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
        ] + ["revealed_Gender"]
        if "human_Gender" in df_beliefs.columns:
            cols += ["human_Gender"]
        demographics[dataset] += cols
        cols.append(id_col[dataset])
        df = df.merge(df_beliefs[cols], on=id_col[dataset])

        # df[
        #     [
        #         c
        #         for c in demographics[dataset]
        #         if "_model_response" in c or "_user_prompt" in c
        #     ]
        # ] = minmax_scale(
        #     df[
        #         [
        #             c
        #             for c in demographics[dataset]
        #             if "_model_response" in c or "_user_prompt" in c
        #         ]
        #     ]
        # )

        cols = [
            domain,
        ]

        for col in cols:
            filtered_df = df.loc[~df[col].isna()]
            if not os.path.isfile(
                f"figures_leace_regression/{dataset}_{concept}_{col}_baseline_1.png"
            ):
                demo_cols = [
                    c
                    for c in demographics[dataset]
                    if "_model_response" in c
                    or "_user_prompt" in c
                    or c == "model_name"
                ]
                mod = smf.ols(
                    formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                )
                res = mod.fit()
                result_df = pd.read_html(
                    res.summary().tables[1].as_html(), header=0, index_col=0
                )[0].reset_index()
                fig = plt.figure(figsize=(6.5, 5))
                ax = sns.barplot(
                    result_df.loc[
                        (result_df["P>|t|"] < 0.05)
                        & (result_df["index"] != "Intercept")
                    ].sort_values(by="coef"),
                    x="index",
                    y="coef",
                )
                ax.tick_params(axis="x", labelrotation=90)
                fig.savefig(
                    f"figures_leace_regression/{dataset}_{concept}_{col}_baseline_1.png",
                    bbox_inches="tight",
                )
                plt.show()
                print(res.summary())
            if "topic" in filtered_df:
                filtered_df = filtered_df.loc[
                    filtered_df.duplicated(subset=["topic"], keep=False)
                ]
            if not os.path.isfile(
                f"figures_leace_regression/{dataset}_{concept}_{col}_baseline_2.png"
            ):
                demo_cols = [
                    c
                    for c in demographics[dataset]
                    if "_model_response" in c
                    or "_user_prompt" in c
                    or c == "topic"
                    or c == "model_name"
                ]
                mod = smf.ols(
                    formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                )
                res = mod.fit()
                result_df = pd.read_html(
                    res.summary().tables[1].as_html(), header=0, index_col=0
                )[0].reset_index()
                result_df["linguistic"] = ~result_df["index"].str.contains("topic")
                fig = plt.figure(figsize=(40, 5))
                ax = sns.barplot(
                    result_df.loc[
                        (result_df["P>|t|"] < 0.05)
                        & (result_df["index"] != "Intercept")
                    ].sort_values(by="coef"),
                    x="index",
                    y="coef",
                    hue="linguistic",
                )
                ax.tick_params(axis="x", labelrotation=90)
                fig.savefig(
                    f"figures_leace_regression/{dataset}_{concept}_{col}_baseline_2.png",
                    bbox_inches="tight",
                )
                plt.show()
                print(res.summary())

            for demographic in [
                "age",
                "gender",
                "education",
                "ethnicity",
                "religion",
                "english",
                "marital",
            ]:
                if not os.path.isfile(
                    f"figures_leace_regression/{dataset}_{concept}_{col}_{demographic}_baseline_3.png"
                ):
                    demo_cols = [
                        c
                        for c in demographics[dataset]
                        if demographic in c.lower()
                        or "_model_response" in c
                        or "_user_prompt" in c
                        or c == "topic"
                        or c == "model_name"
                    ]

                    mod = smf.ols(
                        formula=f"{col} ~ {' + '.join(demo_cols)}", data=filtered_df
                    )
                    res = mod.fit()
                    result_df = pd.read_html(
                        res.summary().tables[1].as_html(), header=0, index_col=0
                    )[0].reset_index()
                    result_df["type"] = result_df["index"].str.extract("(topic)")
                    result_df["type"].loc[
                        result_df["index"].str.contains(demographic)
                    ] = "demographic"
                    result_df["type"].loc[result_df["type"].isna()] = "linguistic"
                    fig = plt.figure(figsize=(45, 5))
                    ax = sns.barplot(
                        result_df.loc[
                            (result_df["P>|t|"] < 0.05)
                            & (result_df["index"] != "Intercept")
                        ].sort_values(by="coef"),
                        x="index",
                        y="coef",
                        hue="type",
                    )
                    ax.tick_params(axis="x", labelrotation=90)
                    fig.savefig(
                        f"figures_leace_regression/{dataset}_{concept}_{col}_{demographic}_baseline_3.png",
                        bbox_inches="tight",
                    )
                    plt.show()
                    print(res.summary())